In [ ]:
import pandas as pd
import numpy as np
import os

# Función para calcular el Awesome Oscillator
def awesome_oscillator(df, short_period=5, long_period=34):
    df['hl2'] = (df['High'] + df['Low']) / 2
    df['ao'] = df['hl2'].rolling(short_period).mean() - df['hl2'].rolling(long_period).mean()
    df['trend'] = np.where(df['ao'] > 0, 'Alcista', 'Bajista')
    return df

# Función para procesar cada archivo
def process_file(file_path):
    data = pd.read_csv(file_path, parse_dates=['at'])
    
    # Renombrar columnas
    data.rename(columns={'at': 'Date', 'open': 'Open', 'max': 'High', 'min': 'Low', 'close': 'Close'}, inplace=True)
    data.set_index('Date', inplace=True)
    
    # Calcular Awesome Oscillator
    data = awesome_oscillator(data)
    
    # Calcular medias móviles simples (SMA) para SSL Channel
    period = 10
    data['smaHigh'] = data['High'].rolling(window=period).mean()
    data['smaLow'] = data['Low'].rolling(window=period).mean()
    
    # Inicialización de Hlv
    data['Hlv'] = np.nan
    
    # Calcular Hlv para SSL Channel
    for i in range(period, len(data)):
        if data['Close'].iloc[i] > data['smaHigh'].iloc[i]:
            data['Hlv'].iloc[i] = 1
        elif data['Close'].iloc[i] < data['smaLow'].iloc[i]:
            data['Hlv'].iloc[i] = -1
        else:
            data['Hlv'].iloc[i] = data['Hlv'].iloc[i - 1]
    
    # Calcular sslDown y sslUp para SSL Channel
    data['sslDown'] = np.where(data['Hlv'] < 0, data['smaHigh'], data['smaLow'])
    data['sslUp'] = np.where(data['Hlv'] < 0, data['smaLow'], data['smaHigh'])
    
    # Detectar cruces para SSL Channel
    data['cross'] = np.where(
        ((data['sslDown'].shift(1) < data['sslUp'].shift(1)) & (data['sslDown'] > data['sslUp'])) |
        ((data['sslDown'].shift(1) > data['sslUp'].shift(1)) & (data['sslDown'] < data['sslUp'])),
        1, 0
    )
    
    # Determinar tipo de cruce para SSL Channel
    data['cross_type'] = np.where(
        (data['sslDown'].shift(1) < data['sslUp'].shift(1)) & (data['sslDown'] > data['sslUp']),
        'Bajista',
        np.where((data['sslDown'].shift(1) > data['sslUp'].shift(1)) & (data['sslDown'] < data['sslUp']),
                 'Alcista', np.nan)
    )
    
    # Crear DataFrame para almacenar los resultados de los cruces para SSL Channel
    crosses = data[data['cross'] == 1].copy()
    crosses['close_at_cross'] = crosses['Close']
    
    # Determinar dirección del cierre de la vela
    data['direccion_vela'] = np.where(data['Close'] > data['Open'], 'Alcista', 'Bajista')
    
    # Determinar posición del Awesome Oscillator
    data['ao_position'] = np.where(data['ao'] > 0, 'Alcista', 'Bajista')
    
    # Determinar posición del SSL
    data['ssl_position'] = np.where(data['Hlv'] > 0, 'Alcista', 'Bajista')
    
    # Calcular la media móvil de 50 periodos
    data['sma50'] = data['Close'].rolling(window=50).mean()
    
    # Crear columna de operación considerando la media móvil de 50 periodos
    data['operacion'] = np.where(
        (data['ao_position'] == data['ssl_position']) & (data['Close'] > data['sma50']),
        'Alcista',
        np.where(
            (data['ao_position'] == data['ssl_position']) & (data['Close'] < data['sma50']),
            'Bajista',
            'Discrepancia'
        )
    )
    
    # Calcular el resultado según las condiciones dadas
    data['resultado'] = np.where(
        data['operacion'] == data['direccion_vela'], 
        85,
        np.where(data['operacion'] == 'Discrepancia', 'Discrepancia', -100)
    )
    
    # Guardar DataFrame en archivo CSV
    output_file_path = os.path.join(output_folder, os.path.basename(file_path))
    data.to_csv(output_file_path)
    
    # Calcular el número de discrepancias y resultados
    discrepancias = data[data['resultado'] == 'Discrepancia']['resultado'].count()
    resultado_85 = data[data['resultado'] == 85]['resultado'].count()
    resultado_100 = data[data['resultado'] == -100]['resultado'].count()
    
    # Encontrar ocurrencias consecutivas de 85
    consecutivos_85 = (data['resultado'] == 85).astype(int).groupby((data['resultado'] != 85).cumsum()).cumsum()
    
    # Contar las veces que aparecen 2, 3 y 4 veces consecutivas
    veces_consecutivas_85 = consecutivos_85.value_counts()
    
    # Encontrar ocurrencias consecutivas de -100
    consecutivos_100 = (data['resultado'] == -100).astype(int).groupby((data['resultado'] != -100).cumsum()).cumsum()
    
    # Contar las veces que aparecen 2, 3 y 4 veces consecutivas
    veces_consecutivas_100 = consecutivos_100.value_counts()
    
    # Calcular el total de pérdida y el mínimo a ganar con el profit base
    total_perdida = resultado_100 * 100
    minimo_ganar = resultado_85 * 85
    
    # Calcular el interés compuesto con capital inicial
    capital_inicial = 100
    tasa_interes = 0.85
    
    ciclos = [2, 3, 4]
    racha_85 = [veces_consecutivas_85.get(2, 0), veces_consecutivas_85.get(3, 0), veces_consecutivas_85.get(4, 0)]
    
    racha_100 = [veces_consecutivas_100.get(2, 0), veces_consecutivas_100.get(3, 0), veces_consecutivas_100.get(4, 0)]
    
    interes_compuesto = []
    
    for ciclo, factor in zip(ciclos, racha_85):
        capital_final = capital_inicial * (1 + tasa_interes) ** ciclo * factor
        interes_compuesto.append(capital_final)
    
    # Crear DataFrame con los resultados
    resultados = pd.DataFrame({
        'Archivo': [os.path.basename(file_path)],
        'Número de discrepancias': [discrepancias],
        'Número de resultados 85': [resultado_85],
        'Número de -100': [resultado_100],
        'Número de veces que aparece 85 2 veces consecutivas': [veces_consecutivas_85.get(2, 0)],
        'Número de veces que aparece 85 3 veces consecutivas': [veces_consecutivas_85.get(3, 0)],
        'Número de veces que aparece 85 4 veces consecutivas': [veces_consecutivas_85.get(4, 0)],
        'Número de veces que aparece -100 2 veces consecutivas': [veces_consecutivas_100.get(2, 0)],
        'Número de veces que aparece -100 3 veces consecutivas': [veces_consecutivas_100.get(3, 0)],
        'Número de veces que aparece -100 4 veces consecutivas': [veces_consecutivas_100.get(4, 0)],
        'Total de pérdida': [total_perdida],
        'Mínimo a ganar con el profit base': [minimo_ganar],
        'Interés compuesto a un 85% para 2 ciclos': [interes_compuesto[0]],
        'Interés compuesto a un 85% para 3 ciclos': [interes_compuesto[1]],
        'Interés compuesto a un 85% para 4 ciclos': [interes_compuesto[2]]
    })
    
    return resultados

# Carpeta que contiene los archivos CSV
input_folder = 'D:\\iqrobot\\df_test\\eurusd\\5m'
output_folder = 'D:\\iqrobot\\df_test\\eurusd\\5m\\processed'

# Crear la carpeta de salida si no existe
os.makedirs(output_folder, exist_ok=True)

# Lista para almacenar los resultados de todos los archivos
all_results = []

# Iterar sobre todos los archivos CSV en la carpeta
for file_name in os.listdir(input_folder):
    if file_name.endswith('.csv'):
        file_path = os.path.join(input_folder, file_name)
        result = process_file(file_path)
        all_results.append(result)

# Concatenar todos los resultados en un solo DataFrame
final_results = pd.concat(all_results, ignore_index=True)

# Guardar los resultados en un único archivo CSV
final_results.to_csv(r'D:\iqrobot\resultados_finales.csv', index=False)


In [ ]:
import os
import pandas as pd

# Función para calcular la Media Móvil
def calcular_media_movil(data, window=50):
    return data['close'].rolling(window=window).mean()

# Función para calcular el SSL Channel
def calcular_ssl_channel(data, window=12):
    high_max = data['max'].rolling(window=window).max()
    low_min = data['min'].rolling(window=window).min()
    ssl_up = high_max + low_min / 2
    ssl_down = high_max - low_min / 2
    return ssl_up, ssl_down

# Función para calcular el Awesome Oscillator
def calcular_awesome_oscillator(data):
    return data['close'].rolling(window=5).mean() - data['close'].rolling(window=34).mean()

# Función para aplicar las condiciones dadas y calcular los resultados
def calcular_resultados(data):
    # Calcular indicadores
    ma = calcular_media_movil(data)
    ssl_up, ssl_down = calcular_ssl_channel(data)
    ao = calcular_awesome_oscillator(data)
    
    # Inicializar lista para almacenar resultados
    resultados = []
    
    # Iterar sobre los datos
    for i in range(len(data)):
        # Verificar condiciones para cruce bajista
        if ssl_up[i] < ssl_down[i] and ma[i] > data['close'][i] and ao[i] < 0:
            # Tomar la siguiente vela y restarla a la tercera vela
            if i + 3 < len(data):
                resultado = data['close'][i + 3] - data['close'][i + 1]
                if resultado < 0:
                    resultados.append(85)
                else:
                    resultados.append(100)
            else:
                resultados.append(None)
        # Verificar condiciones para cruce alcista
        elif ssl_up[i] > ssl_down[i] and ma[i] < data['close'][i] and ao[i] > 0:
            # Tomar la siguiente vela y restarla a la tercera vela
            if i + 3 < len(data):
                resultado = data['close'][i + 3] - data['close'][i + 1]
                if resultado < 0:
                    resultados.append(100)
                else:
                    resultados.append(85)
            else:
                resultados.append(None)
        else:
            resultados.append(None)
    
    return resultados

# Directorio de los archivos
directory = r'D:\iqrobot\df_test\eurusd\5m'

# Lista para almacenar los DataFrames de cada archivo
dfs = []

# Iterar sobre los archivos en el directorio
for filename in os.listdir(directory):
    if filename.endswith('.csv'):
        filepath = os.path.join(directory, filename)
        # Cargar el archivo CSV como DataFrame
        df = pd.read_csv(file_path)
        df['at'] = pd.to_datetime(df['at']) 
        # Calcular los resultados y añadirlos como una columna al DataFrame
        df['resultado'] = calcular_resultados(df)
        # Agregar el DataFrame a la lista
        dfs.append(df)

# Concatenar todos los DataFrames en uno solo
merged_df = pd.concat(dfs)

# Guardar el DataFrame combinado en un archivo CSV
merged_df.to_csv(r'D:\iqrobot\resultados_combinados.csv', index=False)


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def awesome_oscillator(df, short_period=5, long_period=34):
    df['hl2'] = (df['High'] + df['Low']) / 2
    df['ao'] = df['hl2'].rolling(short_period).mean() - df['hl2'].rolling(long_period).mean()
    df['trend'] = np.where(df['ao'] > 0, 'Alcista', 'Bajista')
    return df

# Cargar los datos desde un archivo CSV
data = pd.read_csv('D:\\iqrobot\\df_test\\eurusd\\5m\\EURUSD_5m_20240101_20240624.csv', parse_dates=['at'])

# Asegúrate de que las columnas coincidan con los nombres utilizados en el DataFrame
data.rename(columns={'at': 'Date', 'open': 'Open', 'max': 'High', 'min': 'Low', 'close': 'Close'}, inplace=True)
data.set_index('Date', inplace=True)

# Cálculo del indicador Awesome Oscillator
data = awesome_oscillator(data)

# Cálculo de las medias móviles simples (SMA) para SSL Channel
period = 10
data['smaHigh'] = data['High'].rolling(window=period).mean()
data['smaLow'] = data['Low'].rolling(window=period).mean()

# Inicialización de Hlv
data['Hlv'] = np.nan

# Cálculo del Hlv para SSL Channel
for i in range(period, len(data)):
    if data['Close'].iloc[i] > data['smaHigh'].iloc[i]:
        data['Hlv'].iloc[i] = 1
    elif data['Close'].iloc[i] < data['smaLow'].iloc[i]:
        data['Hlv'].iloc[i] = -1
    else:
        data['Hlv'].iloc[i] = data['Hlv'].iloc[i - 1]

# Cálculo de sslDown y sslUp para SSL Channel
data['sslDown'] = np.where(data['Hlv'] < 0, data['smaHigh'], data['smaLow'])
data['sslUp'] = np.where(data['Hlv'] < 0, data['smaLow'], data['smaHigh'])

# Detectar cruces para SSL Channel
data['cross'] = np.where((data['sslDown'].shift(1) < data['sslUp'].shift(1)) & (data['sslDown'] > data['sslUp']) |
                         (data['sslDown'].shift(1) > data['sslUp'].shift(1)) & (data['sslDown'] < data['sslUp']), 1, 0)

# Determinar tipo de cruce para SSL Channel
data['cross_type'] = np.where((data['sslDown'].shift(1) < data['sslUp'].shift(1)) & (data['sslDown'] > data['sslUp']),
                              'Bajista',
                              np.where((data['sslDown'].shift(1) > data['sslUp'].shift(1)) & (data['sslDown'] < data['sslUp']),
                                       'Alcista', np.nan))

# Crear la figura de Plotly
fig = go.Figure()

# Añadir las velas OHLC
fig.add_trace(go.Candlestick(
    x=data.index,
    open=data['Open'],
    high=data['High'],
    low=data['Low'],
    close=data['Close'],
    name='OHLC'
))

# Añadir sslDown para SSL Channel
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['sslDown'],
    mode='lines',
    line=dict(color='red', width=2),
    name='sslDown'
))

# Añadir sslUp para SSL Channel
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['sslUp'],
    mode='lines',
    line=dict(color='lime', width=2),
    name='sslUp'
))


# Configurar el diseño
fig.update_layout(
    title='SSL Channel with Awesome Oscillator',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False
)

# Mostrar la figura
fig.show()


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def awesome_oscillator(df, short_period=5, long_period=34):
    df['hl2'] = (df['High'] + df['Low']) / 2
    df['ao'] = df['hl2'].rolling(short_period).mean() - df['hl2'].rolling(long_period).mean()
    df['trend'] = np.where(df['ao'] > 0, 'Alcista', 'Bajista')
    return df

# Cargar los datos desde un archivo CSV
data = pd.read_csv('D:\\iqrobot\\df_test\\eurusd\\5m\\EURUSD_5m_20240101_20240624.csv', parse_dates=['at'])

# Asegúrate de que las columnas coincidan con los nombres utilizados en el DataFrame
data.rename(columns={'at': 'Date', 'open': 'Open', 'max': 'High', 'min': 'Low', 'close': 'Close'}, inplace=True)
data.set_index('Date', inplace=True)

# Cálculo del indicador Awesome Oscillator
data = awesome_oscillator(data)

# Cálculo de las medias móviles simples (SMA) para SSL Channel y la nueva media móvil de 50 períodos
short_period = 10
long_period = 100
data['smaHigh'] = data['High'].rolling(window=short_period).mean()
data['smaLow'] = data['Low'].rolling(window=short_period).mean()
data['sma50'] = data['Close'].rolling(window=long_period).mean()

# Inicialización de Hlv
data['Hlv'] = np.nan

# Cálculo del Hlv para SSL Channel
for i in range(long_period, len(data)):
    if data['Close'].iloc[i] > data['smaHigh'].iloc[i]:
        data['Hlv'].iloc[i] = 1
    elif data['Close'].iloc[i] < data['smaLow'].iloc[i]:
        data['Hlv'].iloc[i] = -1
    else:
        data['Hlv'].iloc[i] = data['Hlv'].iloc[i - 1]

# Cálculo de sslDown y sslUp para SSL Channel
data['sslDown'] = np.where(data['Hlv'] < 0, data['smaHigh'], data['smaLow'])
data['sslUp'] = np.where(data['Hlv'] < 0, data['smaLow'], data['smaHigh'])

# Crear la figura de Plotly con subtramas
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05)

# Añadir las velas OHLC a la subtrama superior
fig.add_trace(go.Candlestick(
    x=data.index,
    open=data['Open'],
    high=data['High'],
    low=data['Low'],
    close=data['Close'],
    name='OHLC'
), row=1, col=1)

# Añadir sslDown para SSL Channel a la subtrama superior
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['sslDown'],
    mode='lines',
    line=dict(color='red', width=2),
    name='sslDown'
), row=1, col=1)

# Añadir sslUp para SSL Channel a la subtrama superior
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['sslUp'],
    mode='lines',
    line=dict(color='lime', width=2),
    name='sslUp'
), row=1, col=1)

# Añadir la media móvil de 50 períodos a la subtrama superior
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['sma50'],
    mode='lines',
    line=dict(color='blue', width=2),
    name='SMA 50'
), row=1, col=1)

# Añadir Awesome Oscillator como histograma a la subtrama inferior
colors = {'Alcista': 'green', 'Bajista': 'red'}
fig.add_trace(go.Bar(x=data.index, y=data['ao'], marker_color=data['trend'].map(colors), name='Awesome Oscillator'), row=2, col=1)

# Configurar el diseño
fig.update_layout(
    title='SSL Channel with Awesome Oscillator',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False
)

# Mostrar la figura
fig.show()


In [ ]:
# Crear un nuevo DataFrame para almacenar los resultados
results_df = pd.DataFrame(columns=['operación', 'resultado', 'direccion'])

# Variables para seguimiento de operaciones abiertas
open_trade = False
trade_type = None
entry_index = 0

# Iterar sobre los datos
for i in range(1, len(data)):
    # Verificar si hay una operación abierta y si es momento de cerrar la operación
    if open_trade and (i - entry_index == 3):
        # Cerrar la operación después de 3 velas
        results_df.loc[len(results_df)] = [len(results_df) + 1, 85, trade_type]  # Suponemos que la operación se gana
        open_trade = False
        trade_type = None
    
    # Verificar las condiciones de compra
    if not open_trade:
        if (data['Close'].iloc[i] > data['sma50'].iloc[i]) and (data['ao'].iloc[i] > 1):
            if (data['sslUp'].iloc[i-1] <= data['sslDown'].iloc[i-1]) and (data['sslUp'].iloc[i] > data['sslDown'].iloc[i]):
                # Entrada en compra después de cruce alcista en SSL
                open_trade = True
                trade_type = 'compra'
                entry_index = i

    # Verificar las condiciones de venta
    if not open_trade:
        if (data['Close'].iloc[i] < data['sma50'].iloc[i]) and (data['ao'].iloc[i] < 1):
            if (data['sslUp'].iloc[i-1] >= data['sslDown'].iloc[i-1]) and (data['sslUp'].iloc[i] < data['sslDown'].iloc[i]):
                # Entrada en venta después de cruce bajista en SSL
                open_trade = True
                trade_type = 'venta'
                entry_index = i

# Contar el número de ocurrencias de 85 y 100 en results_df
count_85 = results_df['resultado'].value_counts().get(85, 0)
count_100 = results_df['resultado'].value_counts().get(100, 0)

# Mostrar los resultados
print(results_df)
print("Número de ocurrencias de 85:", count_85)
print("Número de ocurrencias de 100:", count_100)


In [ ]:
# Guardar el DataFrame en un archivo CSV
results_folder_path = r'D:\iqrobot\resultados'
output_file = os.path.join(results_folder_path, 'eurusd_results.csv')
results_df.to_csv(output_file, index=False),,,,,,,,,,,,,,,,,,,,,,,,,,,,,

In [ ]:
Número de ocurrencias de 85:  15119
Número de ocurrencias de 100: 45351

Número de ocurrencias de 85:  12586
Número de ocurrencias de 100: 37752


In [13]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

def awesome_oscillator(df, short_period=5, long_period=34):
    df['hl2'] = (df['High'] + df['Low']) / 2
    df['ao'] = df['hl2'].rolling(short_period).mean() - df['hl2'].rolling(long_period).mean()
    df['trend'] = np.where(df['ao'] > 0, 'Alcista', 'Bajista')
    return df

# Cargar los datos desde un archivo CSV
data = pd.read_csv('D:\\iqrobot\\df_test\\eurusd\\5m\\EURUSD_5m_20240101_20240624.csv', parse_dates=['at'])

# Asegúrate de que las columnas coincidan con los nombres utilizados en el DataFrame
data.rename(columns={'at': 'Date', 'open': 'Open', 'max': 'High', 'min': 'Low', 'close': 'Close'}, inplace=True)
data.set_index('Date', inplace=True)

# Cálculo del indicador Awesome Oscillator
data = awesome_oscillator(data)

# Cálculo de las medias móviles simples (SMA) para SSL Channel y la nueva media móvil de 50 períodos
short_period = 20
long_period = 50
data['smaHigh'] = data['High'].rolling(window=short_period).mean()
data['smaLow'] = data['Low'].rolling(window=short_period).mean()
data['sma50'] = data['Close'].rolling(window=long_period).mean()

# Inicialización de Hlv
data['Hlv'] = np.nan

# Cálculo del Hlv para SSL Channel
for i in range(long_period, len(data)):
    if data['Close'].iloc[i] > data['smaHigh'].iloc[i]:
        data.at[data.index[i], 'Hlv'] = 1
    elif data['Close'].iloc[i] < data['smaLow'].iloc[i]:
        data.at[data.index[i], 'Hlv'] = -1
    else:
        data.at[data.index[i], 'Hlv'] = data['Hlv'].iloc[i - 1]

# Cálculo de sslDown y sslUp para SSL Channel
data['sslDown'] = np.where(data['Hlv'] < 0, data['smaHigh'], data['smaLow'])
data['sslUp'] = np.where(data['Hlv'] < 0, data['smaLow'], data['smaHigh'])


# Crear un nuevo DataFrame para almacenar los resultados
results_df = pd.DataFrame(columns=['operación', 'resultado', 'direccion'])

# Variables para seguimiento de operaciones abiertas
open_trade = False
trade_type = None
entry_index = 0
entry_price = 0

# Iterar sobre los datos
for i in range(1, len(data)):
    # Verificar si hay una operación abierta y si es momento de cerrar la operación
    if open_trade and (i - entry_index == 3):
        # Determinar si la operación se ganó o perdió
        if trade_type == 'compra':
            if data['Close'].iloc[i] > entry_price:
                result = 85  # Operación ganada
            else:
                result = 100  # Operación perdida
        elif trade_type == 'venta':
            if data['Close'].iloc[i] < entry_price:
                result = 85  # Operación ganada
            else:
                result = 100  # Operación perdida
        # Cerrar la operación
        results_df.loc[len(results_df)] = [len(results_df) + 1, result, trade_type]
        open_trade = False
        trade_type = None
        entry_price = 0
    
    # Verificar las condiciones de compra
    if not open_trade:
        if (data['Close'].iloc[i] > data['sma50'].iloc[i]):
            if (data['sslUp'].iloc[i-1] <= data['sslDown'].iloc[i-1]) and (data['sslUp'].iloc[i] > data['sslDown'].iloc[i]):
                # Entrada en compra después de cruce alcista en SSL
                open_trade = True
                trade_type = 'compra'
                entry_index = i
                entry_price = data['Close'].iloc[i]

    # Verificar las condiciones de venta
    if not open_trade:
        if (data['Close'].iloc[i] < data['sma50'].iloc[i]):
            if (data['sslUp'].iloc[i-1] >= data['sslDown'].iloc[i-1]) and (data['sslUp'].iloc[i] < data['sslDown'].iloc[i]):
                # Entrada en venta después de cruce bajista en SSL
                open_trade = True
                trade_type = 'venta'
                entry_index = i
                entry_price = data['Close'].iloc[i]

# Contar el número de ocurrencias de 85 y 100 en results_df
count_85 = results_df['resultado'].value_counts().get(85, 0)
count_100 = results_df['resultado'].value_counts().get(100, 0)

# Mostrar los resultados
print(results_df)
print("Número de ocurrencias de 85:", count_85)
print("Número de ocurrencias de 100:", count_100)

# Guardar el DataFrame en un archivo CSV
results_folder_path = r'D:\iqrobot\resultados'
output_file = os.path.join(results_folder_path, 'eurusd_results.csv')
results_df.to_csv(output_file, index=False)


      operación  resultado direccion
0             1        100     venta
1             2        100    compra
2             3         85     venta
3             4         85     venta
4             5        100    compra
...         ...        ...       ...
1281       1282        100    compra
1282       1283         85    compra
1283       1284         85    compra
1284       1285        100     venta
1285       1286         85    compra

[1286 rows x 3 columns]
Número de ocurrencias de 85: 617
Número de ocurrencias de 100: 669


In [15]:
import pandas as pd
import numpy as np
import os

def awesome_oscillator(df, short_period=5, long_period=34):
    df['hl2'] = (df['High'] + df['Low']) / 2
    df['ao'] = df['hl2'].rolling(short_period).mean() - df['hl2'].rolling(long_period).mean()
    df['trend'] = np.where(df['ao'] > 0, 'Alcista', 'Bajista')
    return df

# Cargar los datos desde un archivo CSV
data = pd.read_csv('D:\\iqrobot\\df_test\\eurusd\\5m\\EURUSD_5m_20240101_20240624.csv', parse_dates=['at'])

# Asegúrate de que las columnas coincidan con los nombres utilizados en el DataFrame
data.rename(columns={'at': 'Date', 'open': 'Open', 'max': 'High', 'min': 'Low', 'close': 'Close'}, inplace=True)
data.set_index('Date', inplace=True)

# Crear un DataFrame para almacenar los resultados de las combinaciones de parámetros
best_results = pd.DataFrame(columns=['short_period', 'long_period', 'count_85', 'count_100'])

# Iterar sobre los valores de short_period y long_period
for short_period in range(1, 101):
    for long_period in range(1, 101):
        temp_data = data.copy()
        
        # Cálculo del indicador Awesome Oscillator
        temp_data = awesome_oscillator(temp_data, short_period, long_period)

        # Cálculo de las medias móviles simples (SMA) para SSL Channel y la nueva media móvil
        temp_data['smaHigh'] = temp_data['High'].rolling(window=short_period).mean()
        temp_data['smaLow'] = temp_data['Low'].rolling(window=short_period).mean()
        temp_data['sma50'] = temp_data['Close'].rolling(window=long_period).mean()

        # Inicialización de Hlv
        temp_data['Hlv'] = np.nan

        # Cálculo del Hlv para SSL Channel
        for i in range(long_period, len(temp_data)):
            if temp_data['Close'].iloc[i] > temp_data['smaHigh'].iloc[i]:
                temp_data.at[temp_data.index[i], 'Hlv'] = 1
            elif temp_data['Close'].iloc[i] < temp_data['smaLow'].iloc[i]:
                temp_data.at[temp_data.index[i], 'Hlv'] = -1
            else:
                temp_data.at[temp_data.index[i], 'Hlv'] = temp_data['Hlv'].iloc[i - 1]

        # Cálculo de sslDown y sslUp para SSL Channel
        temp_data['sslDown'] = np.where(temp_data['Hlv'] < 0, temp_data['smaHigh'], temp_data['smaLow'])
        temp_data['sslUp'] = np.where(temp_data['Hlv'] < 0, temp_data['smaLow'], temp_data['smaHigh'])

        # Crear un DataFrame para almacenar los resultados
        results_df = pd.DataFrame(columns=['operación', 'resultado', 'direccion'])

        # Variables para seguimiento de operaciones abiertas
        open_trade = False
        trade_type = None
        entry_index = 0
        entry_price = 0

        # Iterar sobre los datos
        for i in range(1, len(temp_data)):
            # Verificar si hay una operación abierta y si es momento de cerrar la operación
            if open_trade and (i - entry_index == 3):
                # Determinar si la operación se ganó o perdió
                if trade_type == 'compra':
                    if temp_data['Close'].iloc[i] > entry_price:
                        result = 85  # Operación ganada
                    else:
                        result = 100  # Operación perdida
                elif trade_type == 'venta':
                    if temp_data['Close'].iloc[i] < entry_price:
                        result = 85  # Operación ganada
                    else:
                        result = 100  # Operación perdida
                # Cerrar la operación
                results_df.loc[len(results_df)] = [len(results_df) + 1, result, trade_type]
                open_trade = False
                trade_type = None
                entry_price = 0
            
            # Verificar las condiciones de compra
            if not open_trade:
                if (temp_data['Close'].iloc[i] > temp_data['sma50'].iloc[i]):
                    if (temp_data['sslUp'].iloc[i-1] <= temp_data['sslDown'].iloc[i-1]) and (temp_data['sslUp'].iloc[i] > temp_data['sslDown'].iloc[i]):
                        # Entrada en compra después de cruce alcista en SSL
                        open_trade = True
                        trade_type = 'compra'
                        entry_index = i
                        entry_price = temp_data['Close'].iloc[i]

            # Verificar las condiciones de venta
            if not open_trade:
                if (temp_data['Close'].iloc[i] < temp_data['sma50'].iloc[i]):
                    if (temp_data['sslUp'].iloc[i-1] >= temp_data['sslDown'].iloc[i-1]) and (temp_data['sslUp'].iloc[i] < temp_data['sslDown'].iloc[i]):
                        # Entrada en venta después de cruce bajista en SSL
                        open_trade = True
                        trade_type = 'venta'
                        entry_index = i
                        entry_price = temp_data['Close'].iloc[i]

        # Contar el número de ocurrencias de 85 y 100 en results_df
        count_85 = results_df['resultado'].value_counts().get(85, 0)
        count_100 = results_df['resultado'].value_counts().get(100, 0)

        # Crear una fila de resultados y agregarla al DataFrame de mejores resultados
        new_row = pd.DataFrame({'short_period': [short_period], 'long_period': [long_period], 'count_85': [count_85], 'count_100': [count_100]})
        best_results = pd.concat([best_results, new_row], ignore_index=True)

# Encontrar la mejor combinación de parámetros
best_combination = best_results.loc[best_results['count_85'] == best_results['count_85'].max()]
best_combination = best_combination.loc[best_combination['count_100'] == best_combination['count_100'].min()]

# Mostrar los resultados
print(best_combination)

# Guardar el DataFrame de mejores resultados en un archivo CSV
results_folder_path = r'D:\iqrobot\resultados'
output_file = os.path.join(results_folder_path, 'best_results.csv')
best_results.to_csv(output_file, index=False)


C:\Users\derek\AppData\Local\Temp\ipykernel_5064\3877739284.py:101: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\derek\AppData\Local\Temp\ipykernel_5064\3877739284.py:102: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\derek\AppData\Local\Temp\ipykernel_5064\3877739284.py:101: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\derek\AppData\Local\Temp\ipykernel_5064\3877739284.py:102: FutureWarning:

Series.__g

KeyboardInterrupt: 